In [17]:
!pip install nltk
!pip install Sastrawi
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Library

In [18]:
import pandas as pd
import re
import string
import os
import requests
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

True

# Import Data from Github

In [20]:
API_URL = "https://api.github.com/repos/maxrumbo/CapstoneProject_CC26-PSU329/contents/data-scientist/kategorisasi/Raw Dataset Pengeluaran?ref=development"

response = requests.get(API_URL)
files = response.json()

csv_files = [f for f in files if f["name"].endswith(".csv")]

print(f"Total CSV: {len(csv_files)}")

Total CSV: 6


# Preprocessing Data Function

In [21]:
def extract_label(filename):
    # contoh: "Raw Dataset - Entertainment.csv" → "Entertainment"
    label = re.sub(r"Raw Dataset\s*-\s*", "", filename)
    label = label.replace(".csv", "").strip()
    return label

In [22]:
# stemmer indo
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stopwords indonesia dari NLTK
stop_words = set(stopwords.words('indonesian'))

# custom stopwords promosi & e-commerce (diperluas)
custom_stopwords = {
    # kata promosi umum
    "promo", "promosi", "sale", "diskon", "discount", "obral",
    "cuci", "gudang", "clearance", "flash", "live", "hot", "deal", "hemat", "murah meriah",

    # kata harga & pembayaran
    "murah", "termurah", "terjangkau", "hemat", "harga",
    "cod", "bayar", "ditempat", "gratis", "free",
    "ongkir", "ongkos", "kirim", "cashback", "voucher",

    # kata kualitas marketing
    "terbaru", "terlaris", "terpercaya", "terpopuler",
    "best", "seller", "bestseller", "hits", "viral",
    "premium", "super", "ultra", "spesial", "eksklusif",
    "original", "ori", "asli", "import", "branded",
    "recommended", "rekomendasi", "pilihan", "unggulan",

    # kata stok & ketersediaan
    "ready", "stock", "stok", "available", "tersedia",
    "new", "arrival", "limited", "edition",

    # kata fashion tidak spesifik
    "fashion", "distro", "brand", "koleksi",
    "kualitas", "berkualitas", "garansi", "resmi",

    # kata ajakan beli
    "buruan", "segera", "borong", "grosir", "ecer",
    "bisa", "dapat",

    # kata sifat umum (1-100)
    "tampan", "cantik", "jelek", "kaya", "miskin",
    "lapar", "haus", "kenyang", "luas", "lebar",
    "sempit", "dekat", "jauh", "manis", "pedas",
    "asin", "asam", "baik", "ramah", "sombong",
    "congkak", "hangat", "dingin", "panjang", "pendek",
    "lurus", "bengkok", "keriting", "rapi", "kotor",
    "bersih", "mancung", "pesek", "pelit", "rakus",
    "buas", "jinak", "malas", "rajin", "produktif",
    "abadi", "positif", "negatif", "religius", "aktif",
    "pasif", "cocok", "setuju", "jahat", "pintar",
    "bodoh", "dendam", "pendiam", "riang", "mulia",
    "indah", "buruk", "agung", "rusak", "benar",
    "salah", "sinis", "aktual", "muda", "remaja",
    "tua", "lucu", "jenaka", "bijaksana", "serius",
    "lancar", "macet", "cemas", "tenang", "putih",
    "hitam", "merah", "biru", "kuning", "hijau",
    "ungu", "abu-abu", "oranye", "cokelat", "kelabu",
    "mendung", "cerah", "terang", "basah", "lembap",
    "kering", "tegang", "seram", "lemah", "lesu",
    "lelah", "pucat", "segar", "kuat",

    # kata sifat umum (101-200)
    "sehat", "waspada", "teledor", "lalai", "teliti",
    "legal", "ilegal", "absolut", "resmi", "sadar",
    "nyaman", "risi", "jijik", "enak", "sedap",
    "harum", "wangi", "adil", "curang", "seimbang",
    "mungil", "elok", "molek", "imut", "tipis",
    "tebal", "halus", "lembut", "kasar", "keras",
    "lunak", "empuk", "lembek", "padat", "kecil",
    "besar", "cepat", "tangkas", "lambat", "lamban",
    "lengket", "lekat", "curam", "terjal", "dalam",
    "dangkal", "manja", "mandiri", "puas", "kecewa",
    "aman", "busuk", "basi", "layu", "sempurna",
    "cacat", "utuh", "penuh", "kosong", "hampa",
    "taat", "asli", "palsu", "biasa", "berisik",
    "ricuh", "ribut", "riuh", "gugup", "berani",
    "takut", "cair", "encer", "kental", "merdu",
    "lincah", "tepat", "awal", "terakhir", "timpang",
    "longgar", "ketat", "setia", "utama", "penting",
    "tajam", "tumpul", "tenteram", "gelisah", "khawatir",
    "kurus", "gemuk", "langsing", "gemulai", "kaku",
    "sebentar", "lama", "keren", "mustahil",

    # kata sifat umum (201-300)
    "sopan", "hormat", "nakal", "bandel", "jahil",
    "lugu", "polos", "cekatan", "ideal", "jujur",
    "bohong", "dusta", "tinggi", "rendah", "rahasia",
    "gegabah", "setengah", "waras", "gila", "cemerlang",
    "luhur", "anggun", "apik", "suram", "redup",
    "umum", "khusus", "khas", "istimewa", "bebas",
    "lepas", "datar", "populer", "eksis", "gadungan",
    "mahal", "ahli", "mahir", "etis", "eksotis",
    "langka", "iri", "cukup", "lengkap", "lebih",
    "asyik", "tunggal", "pusing", "ceroboh", "cermat",
    "cerdas", "buta", "canggung", "kikuk", "malu",
    "kuno", "mutakhir", "modern", "baru", "ambigu",
    "pasti", "serasi", "sesuai", "konyol", "kokoh",
    "rapuh", "steril", "bahagia", "sedih", "marah",
    "sedu", "banyak", "rendah hati", "lapang dada",
    "tinggi hati", "besar kepala", "keras kepala",
    "hati singa", "otak udang", "panjang tangan",
    "ringan tangan", "kepala batu", "wajah tembok",
    "hati busuk", "rendah diri", "mulut besar",
    "berdarah dingin", "terang benderang", "aman tentram",
    "galak", "tegas", "inovatif", "kreatif", "progresif",
    "amanah", "alami", "kimiawi", "biologis", "abai",
    "giat", "dinamis",

    # kata sifat lanjutan (301-420)
    "abaksial", "abal-abal", "abid", "biotik", "abiotik",
    "abnormal", "aboral", "absah", "absurd", "acak",
    "adaptif", "adhesif", "adiguna", "angot", "antarkelompok",
    "antargolongan", "audio", "ayu", "bahana", "baka",
    "fana", "visual", "bambang", "bandang", "barbur",
    "batal", "baya", "belia", "belok", "beloh",
    "bengkil", "bengkak", "berserak-serak", "berserudi", "besing",
    "biadab", "betah", "bestari", "cabik", "cabul",
    "cabur", "cadel", "caplang", "caruk", "cegukan",
    "celek", "cengang", "ceples", "ceronggah", "ciamik",
    "cogah", "curiga", "damping", "damas", "daria",
    "dedak", "dedikatif", "degil", "delikat", "dempet",
    "dengki", "despotik", "dewana", "diagonal", "dibasa",
    "digital", "digdaya", "disruptif", "edan", "editorial",
    "efektif", "efisien", "egosentris", "eklektik", "eksak",
    "eksploitatif", "ekstraktif", "ekstraparlementer", "ekstrem", "erak",
    "fasih", "fanatik", "fantastis", "fatal", "fasik",
    "feral", "filantropis", "fiktif", "fluktuatif", "ganar",
    "ganda", "ganggang", "gani", "ganjil", "ganteng",
    "gatal", "gecul", "genap", "gencar", "hak",
    "halal", "haram", "harus", "hayati", "honorer",
    "iba", "identik", "idrak", "igau", "inklusif",
    "inkognito", "inspiratif", "instan", "jago", "janat",
    "jereng", "jurnalistik",

    # satuan & ukuran
    "gr", "gram", "kg", "liter", "ml", "cm", "mm", "meter",
    "pcs", "pc", "buah", "butir", "biji", "lembar", "helai",
    "pasang", "set", "lusin", "kodi", "roll", "dus", "karton",
    "netto", "neto", "bruto", "gross", "net",
    "xgr", "xkg", "xml", "xpcs", "kilogram", 'kilo', 'kaplet',
    'xl','xxl','s','m','l','xs','xxs','5xl','4xl','xxxl','3xl','lxl'

    # varian & pilihan
    "varian", "variant", "variasi", "pilih", "pilihan",
    "warna", "ukuran", "size", "motif", "model", "tipe", "type",
    "mix", "campur", "acak", "random",

    # kemasan & packaging
    "paket", "packet", "pack", "packing", "kemas", "kemasan",
    "box", "kotak", "botol", "sachet", "pouch", "wrapper",
    "wadah", "tempat", "isi", "berisi", "konten", "content",

    # porsi & menu
    "porsi", "serving", "menu", "reguler", "regular", "large",
    "small", "medium", "mini", "jumbo",

    # bahan & komposisi
    "bahan", "material", "ingredient", "komposisi", "terbuat",
    "berbahan", "mengandung", "kandungan",

    # kata jual & transaksi
    "jual", "dijual", "jualin", "jualan", "seller", "toko",
    "shop", "store", "lapak", "online", "reseller", "dropship",
    "grosir", "ecer", "partai",

    # demografis & usia
    "pria", "wanita", "cowok", "cewek", "cewe", "cowo",
    "laki", "perempuan", "lelaki",
    "dewasa", "anak", "remaja", "muda", "tua", "bayi", "balita",
    "lansia", "senior", "junior",

    # kata penghubung tidak penting
    "dll", "dkk", "dsb", "dst", "etc", "dan", "atau",
    "untuk", "dengan", "dari", "ke", "di", "pada",
    "yang", "adalah", "ini", "itu", "juga", "saja",
    "hanya", "sudah", "belum", "ada", "tidak", "bukan",

    # kata deskripsi generik
    "all", "semua", "seluruh", "berbagai", "beragam",
    "lengkap", "komplit", "full", "half", "setengah",
    "per", "tiap", "setiap",

    'nota', 'kode', 'link', 'linj', 'maaf', 'error', 'etalase', 'tolong', 'saya', 
    'kami', 'kamu', 'anda', 'mereka', 'kalian', 'saya', 'aku', 'gue', 'lo',
}

# Baca CSV
url_datakaggle= "https://raw.githubusercontent.com/maxrumbo/CapstoneProject_CC26-PSU329/development/data-scientist/kategorisasi/indonesian-adjective-sentiment-raw.csv"
df = pd.read_csv(url_datakaggle)

# Ambil semua kata dari kolom 'word' dan tambahkan ke custom_stopwords
words_from_csv = set(df['word'].dropna().str.lower().str.strip().tolist())
custom_stopwords.update(words_from_csv)

print(f"Jumlah kata dari CSV: {len(words_from_csv)}")
print(f"Total custom_stopwords sekarang: {len(custom_stopwords)}")

# gabungkan semua stopwords
all_stopwords = stop_words.union(custom_stopwords)

if "solar" in all_stopwords:
    all_stopwords.remove("solar")

def normalize_words(text: str) -> str:
    """Normalisasi kata slang/typo ke bentuk baku."""
    words = text.split()
    normalized = []
    for word in words:
        replacement = normalisasi_kata.get(word, word)
        if replacement:              # skip kata yang di-map ke ""
            normalized.append(replacement)
    return " ".join(normalized)


def clean_text(text: str) -> str:
    """Lowercase, hapus angka, tanda baca, dan spasi berlebih."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def remove_stopwords(text: str) -> str:
    """Hapus stopwords umum + promosi e-commerce."""
    words = text.split()
    filtered = [w for w in words if w not in all_stopwords and len(w) > 1]
    return " ".join(filtered)


def stemming(text: str) -> str:
    """Stemming Sastrawi — ubah ke bentuk dasar Bahasa Indonesia."""
    return stemmer.stem(text)


def preprocess_pipeline(text: str) -> str:
    """Pipeline lengkap: clean → normalisasi → stopwords → stemming."""
    # This function remains for convenience if someone wants to use it directly
    # outside of the main checkpointed process.
    text = clean_text(text)
    text = normalize_words(text)
    text = remove_stopwords(text)
    text = stemming(text)
    return text


def hitung_statistik(original: str, hasil: str) -> dict:
    """Hitung statistik reduksi token."""
    tok_awal  = len(str(original).split())
    tok_akhir = len(str(hasil).split())
    reduksi   = tok_awal - tok_akhir
    persen    = round((reduksi / tok_awal * 100) if tok_awal > 0 else 0, 1)
    return {
        "token_awal"  : tok_awal,
        "token_akhir" : tok_akhir,
        "reduksi"     : reduksi,
        "persen_%"    : persen,
    }

Jumlah kata dari CSV: 3596
Total custom_stopwords sekarang: 4038


In [ ]:
COLUMN_NAME  = "Nama Produk"              
OUTPUT_FILE  = "hasil_preprocessing.csv"  
CHECKPOINT_DIR = "preprocessing_checkpoints" 

def load_all_data():
    files = requests.get(API_URL).json()

    csv_files = [f for f in files if f["name"].endswith(".csv")]

    print(f"Total file ditemukan: {len(csv_files)}")

    all_data = []

    for file in csv_files:
        filename = file["name"]
        url = file["download_url"]

        print(f"\nLoading: {filename}")

        try:
            df = pd.read_csv(url)
        except Exception as e:
            print(f"Error membaca {filename}: {e}")
            continue

        # validasi kolom
        if "Nama Produk" not in df.columns:
            print(f"Kolom 'Nama Produk' tidak ada di {filename}")
            continue

        # ambil hanya kolom yang diperlukan
        df = df[["Nama Produk"]].copy()

        # tambah label dari nama file
        df["label"] = extract_label(filename)

        all_data.append(df)

    # gabungkan semua data
    final_df = pd.concat(all_data, ignore_index=True)

    print(f"\nTotal data gabungan: {len(final_df)}")
    print(f"Kolom final: {list(final_df.columns)}")

    return final_df

def main():
    df = load_all_data()

    print(f" Data berhasil dimuat: {len(df)} baris")

    # =========================
    # DEDUPLICATION
    # =========================
    jumlah_sebelum = len(df)
    df = df.drop_duplicates(subset=[COLUMN_NAME], keep='first').reset_index(drop=True)
    jumlah_duplikat = jumlah_sebelum - len(df)

    if jumlah_duplikat > 0:
        print(f" Duplikat ditemukan dan dihapus: {jumlah_duplikat} baris")
    else:
        print(f" Tidak ada duplikat ditemukan. Total data: {len(df)} baris")

    # =========================
    # CHECKPOINT SETUP
    # =========================
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Direktori checkpoint '{CHECKPOINT_DIR}' siap digunakan")

    print(" Menjalankan preprocessing pipeline dengan checkpointing...")

    # =========================
    # STEP 1: CLEAN TEXT
    # =========================
    print(" Menerapkan: Cleaning text...")

    df["text_cleaned"] = df[COLUMN_NAME].apply(clean_text)

    checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_1_cleaned.csv")
    df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

    print(f" Checkpoint 1 disimpan: {checkpoint_path}")

    # =========================
    # STEP 2: STOPWORDS
    # =========================
    print(" Menerapkan: Penghapusan stopwords...")

    df["text_stopwords_removed"] = df["text_cleaned"].apply(remove_stopwords)

    checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_2_stopwords.csv")
    df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

    print(f" Checkpoint 2 disimpan: {checkpoint_path}")

    # =========================
    # STEP 3: STEMMING
    # =========================
    print(" Menerapkan: Stemming...")

    df["clean_text"] = df["text_stopwords_removed"].apply(stemming)

    checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_3_stemming.csv")
    df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

    print(f" Checkpoint 3 disimpan: {checkpoint_path}")

    print(" Preprocessing pipeline selesai")

    # =========================
    # STATISTIK PER BARIS
    # =========================
    stats = df.apply(
        lambda row: hitung_statistik(row[COLUMN_NAME], row["clean_text"]),
        axis=1,
        result_type="expand"
    )

    df = pd.concat([df, stats], axis=1)

    # =========================
    # RINGKASAN
    # =========================
    print("=" * 60)
    print(" RINGKASAN PREPROCESSING")
    print("=" * 60)
    print(f" Total baris diproses : {len(df)}")
    print(f" Rata-rata token awal : {df['token_awal'].mean():.1f}")
    print(f" Rata-rata token akhir: {df['token_akhir'].mean():.1f}")
    print(f" Rata-rata reduksi    : {df['persen_%'].mean():.1f}%")
    print("=" * 60)

    # =========================
    # CLEAN FINAL
    # =========================

    df["clean_text"] = df["clean_text"].fillna("").astype(str).str.strip()

    jumlah_sebelum = len(df)
    df = df[df["clean_text"] != ""].reset_index(drop=True)
    jumlah_kosong = jumlah_sebelum - len(df)

    if jumlah_kosong > 0:
        print(f" Baris kosong dihapus: {jumlah_kosong}")

    jumlah_sebelum = len(df)
    df = df.drop_duplicates(subset=["clean_text"], keep="first").reset_index(drop=True)
    jumlah_duplikat = jumlah_sebelum - len(df)

    if jumlah_duplikat > 0:
        print(f" Duplikat dihapus: {jumlah_duplikat} baris")

    # =========================
    # SAVE FINAL OUTPUT
    # =========================
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f" Hasil final disimpan ke '{OUTPUT_FILE}'")

    # =========================
    # PREVIEW HASIL
    # =========================
    pd.set_option("display.max_colwidth", 60)
    pd.set_option("display.width", 120)

    print(df[[
        "Nama Produk",
        "label",
        "text_cleaned",
        "text_stopwords_removed",
        "clean_text",
        "token_awal",
        "token_akhir",
        "persen_%"
    ]].head(10))


if __name__ == "__main__":
    main()

Total file ditemukan: 6

Loading: Raw Dataset - Entertainment.csv

Loading: Raw Dataset - Kesehatan.csv

Loading: Raw Dataset - Konsumsi.csv

Loading: Raw Dataset - Langganan.csv

Loading: Raw Dataset - Tagihan.csv

Loading: Raw Dataset - Transport.csv

Total data gabungan: 68535
Kolom final: ['Nama Produk', 'label']
 Data berhasil dimuat: 68535 baris
 Duplikat ditemukan dan dihapus: 4208 baris
 Direktori checkpoint 'preprocessing_checkpoints' siap digunakan
 Menjalankan preprocessing pipeline dengan checkpointing...
 Menerapkan: Cleaning text...
 Checkpoint 1 disimpan: preprocessing_checkpoints\checkpoint_1_cleaned.csv
 Menerapkan: Penghapusan stopwords...
 Checkpoint 2 disimpan: preprocessing_checkpoints\checkpoint_2_stopwords.csv
 Menerapkan: Stemming...
 Checkpoint 3 disimpan: preprocessing_checkpoints\checkpoint_3_stemming.csv
 Preprocessing pipeline selesai
 RINGKASAN PREPROCESSING
 Total baris diproses : 64327
 Rata-rata token awal : 10.5
 Rata-rata token akhir: 6.9
 Rata-rata r

# Clean Data

In [27]:
clean_data = pd.read_csv(OUTPUT_FILE)
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60194 entries, 0 to 60193
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Nama Produk             60194 non-null  object 
 1   label                   60194 non-null  object 
 2   text_cleaned            60194 non-null  object 
 3   text_stopwords_removed  60194 non-null  object 
 4   clean_text              60194 non-null  object 
 5   token_awal              60194 non-null  float64
 6   token_akhir             60194 non-null  float64
 7   reduksi                 60194 non-null  float64
 8   persen_%                60194 non-null  float64
dtypes: float64(4), object(5)
memory usage: 4.1+ MB


In [28]:
clean_data = clean_data[['clean_text', 'label']]
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60194 entries, 0 to 60193
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   clean_text  60194 non-null  object
 1   label       60194 non-null  object
dtypes: object(2)
memory usage: 940.7+ KB
